# 02 - Exploratory Analysis and Stationarity Testing

This notebook explores the hourly appliance energy series (seasonal profiles, decomposition) and runs the stationarity workflow (ADF, KPSS, ACF/PACF, differencing).

Reusable logic lives in `src/appliance_energy/plotting.py` and `src/appliance_energy/stationarity.py`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

from appliance_energy.config import TARGET, METRICS_DIR
from appliance_energy.data import load_hourly_data
from appliance_energy.plotting import (
    plot_series_overview,
    plot_seasonal_profiles,
    plot_decomposition,
)
from appliance_energy.stationarity import run_stationarity_analysis

hourly = load_hourly_data()
y = hourly[TARGET]
hourly.head()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,55.000000,35.000000,19.890000,46.502778,19.200000,44.626528,19.790000,44.897778,18.932778,45.738750,...,17.016667,45.446667,6.308333,733.750000,92.000000,6.166667,53.416667,5.050000,26.823044,26.823044
2016-01-11 18:00:00,176.666667,51.666667,19.897778,45.879028,19.268889,44.438889,19.770000,44.863333,18.908333,46.066667,...,16.981667,45.290000,5.941667,734.266667,91.583333,5.416667,40.000000,4.658333,22.324206,22.324206
2016-01-11 19:00:00,173.333333,25.000000,20.495556,52.805556,19.925556,46.061667,20.052222,47.227361,18.969444,47.815556,...,16.902222,45.311389,6.000000,734.791667,89.750000,6.000000,40.000000,4.391667,33.734932,33.734932
2016-01-11 20:00:00,125.000000,35.000000,20.961111,48.453333,20.251111,45.632639,20.213889,47.268889,19.190833,49.227917,...,16.890000,45.118889,6.000000,735.283333,87.583333,6.000000,40.000000,4.016667,25.679642,25.679642
2016-01-11 21:00:00,103.333333,23.333333,21.311667,45.768333,20.587778,44.961111,20.373333,46.164444,19.425556,47.918889,...,16.890000,44.807778,5.833333,735.566667,87.416667,6.000000,40.000000,3.816667,18.826274,18.826274


## Series overview

Full series, one-week zoom, and the distribution of hourly appliance energy use.

In [2]:
plot_series_overview(hourly);

**Observation:** the target is strongly right-skewed (most hours have modest usage, with occasional large spikes), and the one-week zoom already suggests a repeating daily pattern.

## Seasonal profiles

Mean usage by hour-of-day and by day-of-week.

In [3]:
plot_seasonal_profiles(hourly);

**Observation:** usage peaks in the early evening (around 17:00-18:00), consistent with occupants returning home and using appliances. Weekday/weekend differences are visible but weaker than the daily cycle.

## Classical decomposition

In [4]:
decomp = plot_decomposition(hourly)

## Stationarity testing

Run ADF and KPSS on the raw series, the first difference, and the seasonal (24-hour) difference, along with ACF/PACF plots for each.

In [5]:
results = run_stationarity_analysis(y)
results.to_csv(METRICS_DIR / "stationarity_tests.csv", index=False)
results


ADF test - raw series
  statistic = -8.9489
  p-value   = 8.834e-15
  lags used = 29, n = 3260
  critical value 1%: -3.4324
  critical value 5%: -2.8624
  critical value 10%: -2.5672
  => stationary (reject H0)

KPSS test - raw series
  statistic = 0.0384
  p-value   = 0.1
  => stationary (fail to reject H0)


c:\Users\Hp\Downloads\appliance-energy-forecasting\project\src\appliance_energy\stationarity.py:62: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, pvalue, lags, crit = kpss(



ADF test - first difference
  statistic = -16.9526
  p-value   = 9.437e-30
  lags used = 29, n = 3259
  critical value 1%: -3.4324
  critical value 5%: -2.8624
  critical value 10%: -2.5672
  => stationary (reject H0)

KPSS test - first difference
  statistic = 0.0783
  p-value   = 0.1
  => stationary (fail to reject H0)


c:\Users\Hp\Downloads\appliance-energy-forecasting\project\src\appliance_energy\stationarity.py:62: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, pvalue, lags, crit = kpss(



ADF test - seasonal difference (24h)
  statistic = -13.5934
  p-value   = 2.016e-25
  lags used = 25, n = 3240
  critical value 1%: -3.4324
  critical value 5%: -2.8624
  critical value 10%: -2.5672
  => stationary (reject H0)

KPSS test - seasonal difference (24h)
  statistic = 0.0087
  p-value   = 0.1
  => stationary (fail to reject H0)


c:\Users\Hp\Downloads\appliance-energy-forecasting\project\src\appliance_energy\stationarity.py:62: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, pvalue, lags, crit = kpss(


,test,series,statistic,p_value,conclusion
0,ADF,raw series,-8.948888,8.833753e-15,stationary (reject H0)
1,KPSS,raw series,0.038399,1.000000e-01,stationary (fail to reject H0)
2,ADF,first difference,-16.952574,9.437199e-30,stationary (reject H0)
3,KPSS,first difference,0.078322,1.000000e-01,stationary (fail to reject H0)
4,ADF,seasonal difference (24h),-13.593417,2.016273e-25,stationary (reject H0)
5,KPSS,seasonal difference (24h),0.008674,1.000000e-01,stationary (fail to reject H0)


## Summary and modelling implications

- The ADF test rejects the unit-root null on the raw series, but the ACF shows a slowly-decaying, periodic pattern with spikes at lags 24, 48 and 72 - the signature of daily seasonality rather than a stochastic trend.
- This motivates **no ordinary differencing (d = 0)** but **seasonal differencing (D = 1, s = 24)**.
- After seasonal differencing, the ACF spikes at 48 and 72 disappear, confirming that a single seasonal difference removes most of the daily periodicity.
- These conclusions directly justify the SARIMA(X) order used in `04_sarimax_models.ipynb`: `order=(1,0,1)`, `seasonal_order=(1,1,1,24)`.